In [ ]:
import json
import csv
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
labels_to_evaluate = [
    "research_type",
    "result_outcome",
    "affiliation",
    "problem_description",
    "goal_objective",
    "research_method",
    "research_question",
    "hypothesis",
    "prediction",
    "contribution",
    "pseudocode",
    "open_source_code",
    "open_experiment_code",
    "train",
    "validation",
    "test",
    "results",
    "hardware_specification",
    "software_dependencies",
    "experiment_setup",
]

In [ ]:
accuracy_results = {
    "research_type": {"accuracy": 0.0, "f1_score": 0.0},
    "result_outcome": {"accuracy": 0.0, "f1_score": 0.0},
    "affiliation": {"accuracy": 0.0, "f1_score": 0.0},
    "problem_description": {"accuracy": 0.0, "f1_score": 0.0},
    "goal_objective": {"accuracy": 0.0, "f1_score": 0.0},
    "research_method": {"accuracy": 0.0, "f1_score": 0.0},
    "result_outcome": {"accuracy": 0.0, "f1_score": 0.0},
    "affiliation": {"accuracy": 0.0, "f1_score": 0.0},
    "problem_description": {"accuracy": 0.0, "f1_score": 0.0},
    "goal_objective": {"accuracy": 0.0, "f1_score": 0.0},
    "research_method": {"accuracy": 0.0, "f1_score": 0.0},
    "research_question": {"accuracy": 0.0, "f1_score": 0.0},
    "hypothesis": {"accuracy": 0.0, "f1_score": 0.0},
    "prediction": {"accuracy": 0.0, "f1_score": 0.0},
    "contribution": {"accuracy": 0.0, "f1_score": 0.0},
    "pseudocode": {"accuracy": 0.0, "f1_score": 0.0},
    "open_source_code": {"accuracy": 0.0, "f1_score": 0.0},
    "open_experiment_code": {"accuracy": 0.0, "f1_score": 0.0},
    "train": {"accuracy": 0.0, "f1_score": 0.0},
    "validation": {"accuracy": 0.0, "f1_score": 0.0},
    "test": {"accuracy": 0.0, "f1_score": 0.0},
    "results": {"accuracy": 0.0, "f1_score": 0.0},
    "hardware_specification": {"accuracy": 0.0, "f1_score": 0.0},
    "software_dependencies": {"accuracy": 0.0, "f1_score": 0.0},
    "experiment_setup": {"accuracy": 0.0, "f1_score": 0.0},
}

In [ ]:
# Ground truth of the evaluation of the conference papers
evaluations_file = "../../evaluations-fixed-zeros.json"

llm_results_files = [
    "../../SOTA/raw_data/run_01.json",
    "../../SOTA/raw_data/run_02.json",
    "../../SOTA/raw_data/run_03.json",
    "../../SOTA/raw_data/run_04.json",
    "../../SOTA/raw_data/run_05.json",
]

In [ ]:
# Read the ground truth evaluations
with open(evaluations_file, "r") as f:
    ground_truth_raw = json.load(f)

ground_truth = {}

for paper in ground_truth_raw:
    # print(paper)
    ground_truth[str(paper["index"])] = {}
    for label in labels_to_evaluate:
        if label in paper:
            ground_truth[str(paper["index"])][label] = {"result": paper[label]}
        else:
            ground_truth[str(paper["index"])][label] = {"result": None}

In [ ]:
for llm_results_file in llm_results_files:

    print(f"Evaluating results from: {llm_results_file}")

    # Read the results of the LLM evaluation
    with open(llm_results_file, "r") as f:
        llm_results = json.load(f)

    model = llm_results["model"]
    evaluation_seconds = llm_results["evaluation_seconds"]

    labels_evaluated = 0
    input_tokens = 0
    thoughts_tokens = 0
    output_tokens = 0
    cost = 0.0

    results = {}

    # Initialize results dictionary for each label
    for label in labels_to_evaluate:

        results[label] = {"y_true": [], "y_pred": []}

        for paper in llm_results:
            # Check if paper is a string of a paper id (skip metadata)
            if not paper.isdigit():
                continue

            # For affiliation combine collaboration and industry
            if label == "affiliation":
                if llm_results[paper][label]["result"] == 2:
                    llm_results[paper][label]["result"] = 1

                if ground_truth[paper][label]["result"] == 2:
                    ground_truth[paper][label]["result"] = 1

            # Add the ground truth and LLM results to the results dictionary
            results[label]["y_true"].append(ground_truth[paper][label]["result"])
            results[label]["y_pred"].append(llm_results[paper][label]["result"])

            input_tokens += llm_results[paper]["input_tokens"]
            thoughts_tokens += llm_results[paper]["thoughts_tokens"]
            output_tokens += llm_results[paper]["output_tokens"]
            cost += llm_results[paper]["cost"]

            labels_evaluated += 1

    # Calculate the accuracy and F1 score for each label
    run_accuracy_results = accuracy_results.copy()

    for result in results:
        y_true = results[result]["y_true"]
        y_pred = results[result]["y_pred"]

        # Calculate accuracy
        accuracy = accuracy_score(y_true, y_pred)

        # Calculate F1 score
        f1 = f1_score(y_true, y_pred)

        run_accuracy_results[result]["accuracy"] = f"{accuracy * 100:.2f}"
        run_accuracy_results[result]["f1_score"] = f"{f1 * 100:.2f}"

    # Save run_accuracy_results to CSV
    run_name = llm_results_file.split("/")[-1].replace(".json", "")

    with open("accuracy_f1_results.csv", "a", newline="") as csvfile:
        fieldnames = [
            "run_name",
            "model",
            "labels_evaluated",
            "evaluation_seconds",
            "input_tokens",
            "thoughts_tokens",
            "output_tokens",
            "cost",
            "research_type_accuracy",
            "research_type_f1_score",
            "result_outcome_accuracy",
            "result_outcome_f1_score",
            "affiliation_accuracy",
            "affiliation_f1_score",
            "problem_description_accuracy",
            "problem_description_f1_score",
            "goal_objective_accuracy",
            "goal_objective_f1_score",
            "research_method_accuracy",
            "research_method_f1_score",
            "research_question_accuracy",
            "research_question_f1_score",
            "hypothesis_accuracy",
            "hypothesis_f1_score",
            "prediction_accuracy",
            "prediction_f1_score",
            "contribution_accuracy",
            "contribution_f1_score",
            "pseudocode_accuracy",
            "pseudocode_f1_score",
            "open_source_code_accuracy",
            "open_source_code_f1_score",
            "open_experiment_code_accuracy",
            "open_experiment_code_f1_score",
            "train_accuracy",
            "train_f1_score",
            "validation_accuracy",
            "validation_f1_score",
            "test_accuracy",
            "test_f1_score",
            "results_accuracy",
            "results_f1_score",
            "hardware_specification_accuracy",
            "hardware_specification_f1_score",
            "software_dependencies_accuracy",
            "software_dependencies_f1_score",
            "experiment_setup_accuracy",
            "experiment_setup_f1_score",
        ]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        if csvfile.tell() == 0:  # Check if the file is empty to write header
            writer.writeheader()
        writer.writerow(
            {
                **{
                    "run_name": run_name,
                    "model": model,
                    "labels_evaluated": labels_evaluated,
                    "evaluation_seconds": evaluation_seconds,
                    "input_tokens": input_tokens,
                    "thoughts_tokens": thoughts_tokens,
                    "output_tokens": output_tokens,
                    "cost": round(cost, 2),
                },
                **{
                    f"{label}_{metric}": metrics[metric]
                    for label, metrics in run_accuracy_results.items()
                    for metric in ["accuracy", "f1_score"]
                },
            }
        )